1. Introduction
2. Dataset Loading
3. Data Understanding
4. NLP Preprocessing
5. Feature Engineering
6. Model Building
7. Model Evaluation
8. Comparison & Insights
9. Conclusion

# Introduction
This project focuses on building an end-to-end Sentiment Analysis system using NLP techniques and Machine Learning models. The pipeline includes text preprocessing, feature engineering using BoW and TF-IDF, and training multiple ML models such as Logistic Regression, Naive Bayes, and Decision Tree. The performance of each model is evaluated and compared using standard metrics.


In [1]:
# data set loading
import pandas as pd

df = pd.read_csv("Tweets.csv")  # change path if needed
df = df[['text', 'airline_sentiment']]

df.columns = ['review', 'sentiment']
df.head()

,review,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [2]:
#Data Understanding
print("Shape:", df.shape)

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

df.sample(5)

Shape: (14640, 2)

Class Distribution:
sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


,review,sentiment
4467,@SouthwestAir #stepup #makeitright re you best...,neutral
12784,@AmericanAir everything for sorted out. Thanks...,positive
3070,@united -huge kudos to the FO of Sunday's flt ...,positive
7828,@JetBlue Thanks. Still booked our trip 3/13-17...,positive
9617,@USAirways then she had to get her own hotel s...,negative


In [3]:
# install & import
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\S
[nltk_data]     Mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# reusable preprocessing function
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove special characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    return " ".join(words)

# apply preprocessing
df['cleaned_review'] = df['review'].apply(preprocess_text)



In [5]:
# Feature Engineering
# - Bag of Words
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['cleaned_review'])
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_review'])

# - TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_review'])

# encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

In [6]:
# model regression 
# train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

# - Logicstic regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# naive bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Decision tree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [8]:
# Model evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='weighted'),
        "Recall": recall_score(y_true, y_pred, average='weighted'),
        "F1 Score": f1_score(y_true, y_pred, average='weighted')
    }

results = {
    "Logistic Regression": evaluate(y_test, y_pred_lr),
    "Naive Bayes": evaluate(y_test, y_pred_nb),
    "Decision Tree": evaluate(y_test, y_pred_dt)
}

pd.DataFrame(results)

,Logistic Regression,Naive Bayes,Decision Tree
Accuracy,0.795082,0.734973,0.707309
Precision,0.785224,0.752005,0.702274
Recall,0.795082,0.734973,0.707309
F1 Score,0.782982,0.685875,0.704539


# Comparision & Insights
* Logistic Regression performed best due to its efficiency with high-dimensional sparse data.
* Naive Bayes performed well but assumes feature independence.
* Decision Tree showed lower performance due to overfitting on sparse features.
* TF-IDF outperformed BoW because it captures word importance.
* Preprocessing significantly improved model accuracy by removing noise.


# Conclusion
In this project, we built a complete NLP pipeline for sentiment analysis. Among all models, Logistic Regression with TF-IDF features achieved the best performance. This demonstrates the importance of proper preprocessing and feature engineering in NLP tasks.
